In [13]:
# setup_database.py

import sqlite3
import csv
from pathlib import Path


def create_database(db_path: str) -> None:
    """Create SQLite database and tables for the e-commerce dataset."""
    db_file = Path(db_path)
    db_file.parent.mkdir(parents=True, exist_ok=True)

    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()

         # Enable foreign key support: Activer les clés étrangères
        cursor.execute("PRAGMA foreign_keys = ON;")

        # (Optionnel) nettoyer les tables pour pouvoir relancer le script
        # Drop tables if they exist (for easier re-runs during development)
        cursor.execute("DROP TABLE IF EXISTS orders;")
        cursor.execute("DROP TABLE IF EXISTS products;")
        cursor.execute("DROP TABLE IF EXISTS customers;")

        # === TABLE customers ===
        # Primary key: customer_id
        # Constraints: email UNIQUE, name and join_date NOT NULL
        cursor.execute(
            """
            CREATE TABLE IF NOT EXISTS customers (
                customer_id INTEGER PRIMARY KEY,
                name        TEXT NOT NULL,
                email       TEXT NOT NULL UNIQUE,
                city        TEXT,
                join_date   TEXT NOT NULL
            );
            """
        )

        # === TABLE products ===
        # Primary key: product_id
        # Constraints: price and cost NOT NULL
        cursor.execute(
            """
            CREATE TABLE IF NOT EXISTS products (
                product_id INTEGER PRIMARY KEY,
                name       TEXT NOT NULL,
                category   TEXT,
                price      REAL NOT NULL,
                cost       REAL NOT NULL
            );
            """
        )

        # === TABLE orders ===
        # Primary key: order_id
        # Foreign keys: customer_id -> customers, product_id -> products
        # Constraints: quantity and order_date NOT NULL
        cursor.execute(
            """
            CREATE TABLE IF NOT EXISTS orders (
                order_id    INTEGER PRIMARY KEY,
                customer_id INTEGER NOT NULL,
                product_id  INTEGER NOT NULL,
                quantity    INTEGER NOT NULL,
                order_date  TEXT NOT NULL,
                FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
                FOREIGN KEY (product_id)  REFERENCES products(product_id)
            );
            """
        )

        conn.commit()
    print(f"Database created: {db_path}")

    ######################################          ############################

def import_csv(db_path: str, csv_file: str, table_name: str, skip_header: bool = True) -> None:
    """
    Import a CSV file into a database table.

    Assumes:
    - The first row of the CSV is a header row with column names
      matching the table's column names.
    """
    csv_path = Path(csv_file)
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV file not found: {csv_file}")

    with sqlite3.connect(db_path) as conn, open(csv_path, newline="", encoding="utf-8") as f:
        cursor = conn.cursor()
        reader = csv.reader(f)

        # Lire l'en-tête
        # Read header row to build dynamic INSERT statement
        header = next(reader) if skip_header else None
        if header is None:
            raise ValueError("CSV must have a header row for dynamic insert.")

        columns = ", ".join(header)
        placeholders = ", ".join(["?"] * len(header))
        insert_sql = f"INSERT INTO {table_name} ({columns}) VALUES ({placeholders})"

        rows = list(reader)
        cursor.executemany(insert_sql, rows)
        conn.commit()

    print(f"Imported {len(rows)} rows from {csv_file} into {table_name}")


def verify_data(db_path: str) -> None:
    """Print row counts for each table to verify CSV import."""
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        for table in ("customers", "products", "orders"):
            cursor.execute(f"SELECT COUNT(*) FROM {table};")
            count = cursor.fetchone()[0]
            print(f"Table {table}: {count} rows")


def main() -> None:
    """Main function to set up the database."""
    DB_PATH = "data/myproject.db"

    # 1) Créer la base et les tables
    create_database(DB_PATH)

    # 2) Importer les CSV (place-les dans le dossier data/)
    import_csv(DB_PATH, "data/customers.csv", "customers")
    import_csv(DB_PATH, "data/products.csv", "products")
    import_csv(DB_PATH, "data/orders.csv", "orders")

    # 3) Vérifier les données
    print("\n=== Row counts ===")
    verify_data(DB_PATH)


if __name__ == "__main__":
    main()


Database created: data/myproject.db
Imported 30 rows from data/customers.csv into customers
Imported 20 rows from data/products.csv into products
Imported 100 rows from data/orders.csv into orders

=== Row counts ===
Table customers: 30 rows
Table products: 20 rows
Table orders: 100 rows
